In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

In [ ]:
print("GPUs:", tf.config.list_physical_devices('GPU'))

In [ ]:
dataset_path = '/content/dataset'

In [ ]:
print(f"Contents of {dataset_path}: {os.listdir(dataset_path)}")

train_sub_path = os.path.join(dataset_path, 'train')
test_sub_path = os.path.join(dataset_path, 'test')

print(f"Contents of {train_sub_path}: {os.listdir(train_sub_path)}")
print(f"Contents of {test_sub_path}: {os.listdir(test_sub_path)}")

In [ ]:
train_gen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True
)

val_gen = ImageDataGenerator(rescale=1./255)

SEED = 42

In [ ]:
train_data = train_gen.flow_from_directory(
    os.path.join(dataset_path, 'train'),
    target_size=(224, 224),
    batch_size=16,
    class_mode='categorical',
    shuffle=True,
    seed=SEED
)

val_path = os.path.join(dataset_path, 'val')
if not os.path.isdir(val_path):
    print("Warning: './dataset/val' not found. Using './dataset/test' as validation data.")
    val_path = os.path.join(dataset_path, 'test')

val_data = val_gen.flow_from_directory(
    val_path,
    target_size=(224, 224),
    batch_size=16,
    class_mode='categorical',
    shuffle=False
)

print("Class indices:", train_data.class_indices)

In [ ]:
base_model = EfficientNetB0(weights='imagenet', include_top=False, input_shape=(224,224,3))
base_model.trainable = False

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(4, activation='softmax')
])

In [ ]:
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
callbacks = [
    EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=2, min_lr=1e-7),
    ModelCheckpoint('best_brain_tumor_model.keras', monitor='val_accuracy', save_best_only=True)
]

history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=10,
    callbacks=callbacks
)

In [ ]:
base_model.trainable = True

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
 )

history_finetune = model.fit(
    train_data,
    validation_data=val_data,
    epochs=10,
    callbacks=callbacks
 )

In [ ]:
test_gen = ImageDataGenerator(rescale=1./255)

test_data = test_gen.flow_from_directory(
    os.path.join(dataset_path, 'test'),
    target_size=(224, 224),
    batch_size=1,
    shuffle=False,
    class_mode='categorical'
)

loss, acc = model.evaluate(test_data)
print('Test Accuracy:', acc)

pred_probs = model.predict(test_data)
pred_classes = np.argmax(pred_probs, axis=1)
true_classes = test_data.classes

cm = tf.math.confusion_matrix(true_classes, pred_classes).numpy()
print('Confusion Matrix:\n', cm)

In [ ]:
train_acc = history.history['accuracy'] + history_finetune.history['accuracy']
val_acc = history.history['val_accuracy'] + history_finetune.history['val_accuracy']

plt.figure(figsize=(8, 5))
plt.plot(train_acc, label='Train Accuracy')
plt.plot(val_acc, label='Validation Accuracy')
plt.title('Model Accuracy Across Both Training Stages')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(alpha=0.2)
plt.show()

In [ ]:
model.save('brain_tumor_model.h5')